# Stage 2 Benchmark — SoccerNet-GSR v1.3

**v1.3 — crash-safe checkpoint/resume + live console progress.** Logic evaluator remains in `stage2_entities/benchmark/`; the notebook only orchestrates the CLI.


In [ ]:
from pathlib import Path
import json, subprocess, sys
PROJECT_DIR = Path.cwd().resolve()
SOCCERNET_ROOT = Path(r"D:\Datasets\SoccerNetGS")
SPLIT = "valid"
SST = PROJECT_DIR / "weights" / "model.pth"
RTMW = PROJECT_DIR / "weights" / "rtmw_l_384x288.onnx"
OUT_QUICK = PROJECT_DIR / "benchmark_results" / "quick"
OUT_FULL = PROJECT_DIR / "benchmark_results" / "full_valid"
print(PROJECT_DIR)

## 0. Checkpoint / resume

Benchmark tự checkpoint sau từng frame perception, từng detection-window và từng tracking ablation. Nếu bị interrupt, chạy lại đúng command với cùng `--output-dir`; các unit đã hoàn thành sẽ hiện `[RESUME]` và không bị tính lại.

`--restart-evaluation` chỉ xóa checkpoint evaluator, **không xóa SST+RTMW perception cache**. `--overwrite-perception` mới là tùy chọn cố ý chạy lại model.


In [ ]:
# Có thể chạy cell này bất kỳ lúc nào, không load SST/RTMW.
if (OUT_QUICK / "checkpoints" / "benchmark_state.json").exists():
    subprocess.run([sys.executable,"benchmark_stage2.py","status","--output-dir",str(OUT_QUICK)],check=True)
else:
    print("Quick benchmark chưa có checkpoint.")


## 1. Inspect dataset
Chạy bước này trước GPU inference. Loader phải thấy `Labels-GameState.json`, version >=1.3, FPS/resolution và nguồn frame/video.

In [ ]:
subprocess.run([sys.executable,"benchmark_stage2.py","inspect","--soccernet-root",str(SOCCERNET_ROOT),"--split",SPLIT],check=True)

## 1.1. Plan benchmark budget

Không load model. In chính xác các `t0`, window và số unique frame cần SST+RTMW trước khi chạy GPU.


In [ ]:
subprocess.run([sys.executable,"benchmark_stage2.py","plan","--soccernet-root",str(SOCCERNET_ROOT),"--split",SPLIT,"--protocol","quick"],check=True)


## 2. Quick benchmark
Mặc định 10 sequence × 3 target frames, ±1 s. Có cache nên có thể chạy lại evaluator mà không inference SST/RTMW lần nữa.
Console cập nhật sequence/frame/evaluation progress, elapsed time và ETA. `Ctrl+C` an toàn: checkpoint đã commit sẽ được giữ.


In [ ]:
cmd=[sys.executable,"benchmark_stage2.py","run","--soccernet-root",str(SOCCERNET_ROOT),"--split",SPLIT,"--protocol","quick","--sst-checkpoint",str(SST),"--rtmw-model",str(RTMW),"--output-dir",str(OUT_QUICK)]
subprocess.run(cmd,check=True)

In [ ]:
summary=json.loads((OUT_QUICK/"benchmark_summary.json").read_text(encoding="utf-8"))
print(json.dumps({
 "detection": summary["detection"],
 "tracking_candidate": {k:summary["tracking_candidate"].get(k) for k in ["HOTA","DetA","AssA","IDF1","TCR"]},
 "acceptance_gate": summary["acceptance_gate"],
},indent=2))

## 3. Full validation
Chỉ chạy sau khi quick protocol và failure cases đã được kiểm tra. Không dùng test để tune threshold.
Nếu full validation bị dừng giữa chừng, chạy lại cell này với cùng `OUT_FULL` để resume.


In [ ]:
RUN_FULL=False
if RUN_FULL:
    cmd=[sys.executable,"benchmark_stage2.py","run","--soccernet-root",str(SOCCERNET_ROOT),"--split",SPLIT,"--protocol","full","--sst-checkpoint",str(SST),"--rtmw-model",str(RTMW),"--output-dir",str(OUT_FULL)]
    subprocess.run(cmd,check=True)

## v1.3 M1 threshold sweep
After a fresh v1.3 quick run, sweep Player/GK thresholds from cached low-score SST detections. This does not rerun SST or RTMW.


In [ ]:
!python benchmark_stage2.py sweep-thresholds --soccernet-root \"D:\\Datasets\\SoccerNetGS\" --split valid --benchmark-dir \".\\benchmark_results\\quick_v13\" --player-thresholds \"0.30,0.35,0.40,0.45,0.50\" --goalkeeper-thresholds \"0.30,0.35,0.40,0.45,0.50\"


Key v1.3 diagnostics are `CandidatePrecision`, `CandidateRecall`, `AnyDuplicateRate_after`, `AnchorCoverage`, `ConditionalTCR`, and overall `TCR`. Exact Player/GK role is a quality metric rather than a hard Stage-3 eligibility gate.
